# LipSync — Data Exploration & Pipeline Prototyping

This notebook explores the GRID corpus dataset, visualises lip ROI extraction,
and validates the data pipeline before model training.

## Contents
1. Dataset Overview & Statistics
2. Sample Video Visualisation
3. Alignment File Analysis
4. Face Mesh & Lip ROI Extraction Demo
5. Data Loader Validation


In [ ]:
import sys
import os

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = Path(PROJECT_ROOT) / "backend" / "data"
RAW_DIR = DATA_DIR / "raw"
ALIGN_DIR = DATA_DIR / "alignments"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Raw videos exist: {RAW_DIR.exists()}")
print(f"Alignments exist: {ALIGN_DIR.exists()}")
print(f"Processed data exist: {PROCESSED_DIR.exists()}")


## 1. Dataset Overview & Statistics

The GRID corpus contains:
- **34 speakers** (s1–s34, s21 missing)
- **1000 sentences** per speaker
- Constrained grammar: `<command> <color> <preposition> <letter> <digit> <adverb>`
- Example: "bin blue at f two now"


In [ ]:
# GRID Vocabulary
GRID_VOCAB = {
    "command": ["bin", "lay", "place", "set"],
    "color": ["blue", "green", "red", "white"],
    "preposition": ["at", "by", "in", "with"],
    "letter": list("abcdefghijklmnopqrstuvwxyz"),
    "digit": ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"],
    "adverb": ["again", "now", "please", "soon"]
}

print("GRID Vocabulary Structure:")
print("=" * 50)
total_combos = 1
for category, words in GRID_VOCAB.items():
    print(f"  {category:15s}: {len(words):3d} options — {words[:5]}{'...' if len(words) > 5 else ''}")
    total_combos *= len(words)
print(f"\nTotal possible sentences: {total_combos:,}")
print(f"Unique characters in vocab: {sorted(set(''.join(w for ws in GRID_VOCAB.values() for w in ws)))}")


In [ ]:
# Count downloaded data
if RAW_DIR.exists():
    speakers = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
    print(f"Downloaded speakers: {len(speakers)}")
    for sp in speakers:
        videos = list((RAW_DIR / sp).rglob("*.mpg")) + list((RAW_DIR / sp).rglob("*.mp4"))
        print(f"  {sp}: {len(videos)} videos")
else:
    print("No raw videos found. Run the download script first:")
    print("  python -m backend.data.download_grid --speakers s1 s2")


## 2. Sample Video Visualisation

Display sampled frames from a GRID video to check resolution, quality, and FPS.


In [ ]:
def visualise_video(video_path, max_frames=10):
    """Display frames from a video file."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Cannot open: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Video: {video_path.name}")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps}")
    print(f"  Total frames: {total_frames}")
    print(f"  Duration: {total_frames / max(fps, 1):.2f}s")

    # Sample frames evenly
    step = max(1, total_frames // max_frames)
    frames = []
    for i in range(0, total_frames, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if len(frames) >= max_frames:
            break
    cap.release()

    if frames:
        fig, axes = plt.subplots(1, len(frames), figsize=(2 * len(frames), 3))
        if len(frames) == 1:
            axes = [axes]
        for ax, frame in zip(axes, frames):
            ax.imshow(frame)
            ax.axis("off")
        plt.suptitle(f"Sampled frames from {video_path.name}", fontsize=10)
        plt.tight_layout()
        plt.show()

# Try to visualise a sample video
sample_videos = list(RAW_DIR.rglob("*.mpg"))[:1] + list(RAW_DIR.rglob("*.mp4"))[:1]
if sample_videos:
    visualise_video(sample_videos[0])
else:
    print("No videos available yet. Download the GRID corpus first.")


## 3. Alignment File Analysis

Parse GRID `.align` files and verify label extraction.


In [ ]:
from backend.utils.lip_extractor import parse_alignment

# Explore alignment files
align_files = sorted(ALIGN_DIR.rglob("*.align"))[:5]

if align_files:
    print("Sample alignment files:")
    print("=" * 60)
    for af in align_files:
        print(f"\n{af.name}:")
        print(f"  Raw contents:")
        with open(af) as f:
            for line in f:
                print(f"    {line.strip()}")
        label = parse_alignment(af)
        print(f"  → Parsed label: '{label}'")
else:
    print("No alignment files found. Download the GRID corpus first.")
    print("  python -m backend.data.download_grid --speakers s1 s2")


## 4. Face Mesh & Lip ROI Extraction Demo

Test the MediaPipe Face Mesh pipeline on sample video frames.


In [ ]:
from backend.utils.face_mesh import FaceMeshProcessor

def demo_lip_extraction(video_path, num_frames=6):
    """Demonstrate lip ROI extraction on a video."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Cannot open: {video_path}")
        return

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total // num_frames)

    processor = FaceMeshProcessor(static_image_mode=True)

    fig, axes = plt.subplots(2, num_frames, figsize=(3 * num_frames, 5))
    frame_idx = 0

    for i in range(0, total, step):
        if frame_idx >= num_frames:
            break

        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break

        detection = processor.process_frame(frame)
        annotated = processor.draw_landmarks(frame, detection)

        # Top row: annotated frame
        axes[0, frame_idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[0, frame_idx].set_title(f"Frame {i}", fontsize=8)
        axes[0, frame_idx].axis("off")

        # Bottom row: lip ROI
        if detection.lip_roi is not None:
            axes[1, frame_idx].imshow(detection.lip_roi, cmap="gray")
            axes[1, frame_idx].set_title(f"Lip ROI {detection.lip_roi.shape}", fontsize=8)
        else:
            axes[1, frame_idx].text(0.5, 0.5, "No face", ha="center", va="center")
        axes[1, frame_idx].axis("off")

        frame_idx += 1

    cap.release()
    processor.close()

    plt.suptitle(f"Lip Extraction: {video_path.name}", fontsize=11)
    plt.tight_layout()
    plt.show()

# Demo on a sample video
sample_videos = list(RAW_DIR.rglob("*.mpg"))[:1] + list(RAW_DIR.rglob("*.mp4"))[:1]
if sample_videos:
    demo_lip_extraction(sample_videos[0])
else:
    print("No videos available. Download the GRID corpus first.")


## 5. Data Loader Validation

Verify the data loader produces correct shapes and label alignments.


In [ ]:
# Check processed data
if PROCESSED_DIR.exists():
    frame_files = sorted(PROCESSED_DIR.rglob("*_frames.npy"))
    label_files = sorted(PROCESSED_DIR.rglob("*_label.txt"))
    print(f"Processed frame files: {len(frame_files)}")
    print(f"Processed label files: {len(label_files)}")

    if frame_files:
        # Check a sample
        sample = np.load(frame_files[0])
        label_text = open(str(frame_files[0]).replace("_frames.npy", "_label.txt")).read().strip()

        print(f"\nSample: {frame_files[0].name}")
        print(f"  Frames shape: {sample.shape}")
        print(f"  Dtype: {sample.dtype}")
        print(f"  Value range: [{sample.min():.3f}, {sample.max():.3f}]")
        print(f"  Label: '{label_text}'")

        # Visualise sample frames
        fig, axes = plt.subplots(1, 8, figsize=(16, 2))
        step = max(1, sample.shape[0] // 8)
        for idx, ax in enumerate(axes):
            frame_i = min(idx * step, sample.shape[0] - 1)
            ax.imshow(sample[frame_i], cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"f{frame_i}", fontsize=8)
            ax.axis("off")
        plt.suptitle(f"Processed lip ROIs — '{label_text}'", fontsize=10)
        plt.tight_layout()
        plt.show()
else:
    print("No processed data found. Run the extraction pipeline first:")
    print("  python -m backend.utils.lip_extractor")


In [ ]:
# Test the data loader (only if processed data exists)
if PROCESSED_DIR.exists() and list(PROCESSED_DIR.rglob("*_frames.npy")):
    from backend.utils.data_loader import create_dataset

    train_ds, val_ds, n_train, n_val = create_dataset(
        str(PROCESSED_DIR), batch_size=4
    )

    print(f"Train samples: {n_train}")
    print(f"Val samples: {n_val}")

    for frames_batch, labels_batch in train_ds.take(1):
        print(f"\nBatch shapes:")
        print(f"  Frames: {frames_batch.shape}")  # (batch, 75, 50, 100, 1)
        print(f"  Labels: {labels_batch.shape}")   # (batch, 40)
        print(f"  Frame range: [{frames_batch.numpy().min():.3f}, {frames_batch.numpy().max():.3f}]")

        # Decode labels
        from backend.model.train import IDX_TO_CHAR
        for i in range(min(4, labels_batch.shape[0])):
            label = labels_batch[i].numpy()
            decoded = "".join(IDX_TO_CHAR.get(int(c), "") for c in label if c != 0)
            print(f"  Label {i}: '{decoded}'")

    print("\n✅ Data loader validation passed!")
else:
    print("Skipping data loader test — no processed data available.")


## Summary

Phase 1 pipeline status:
- [ ] GRID corpus downloaded
- [ ] Lip ROI extraction producing clean frame sequences
- [ ] Data loader outputting correct shapes and labels
- [ ] Ready for Phase 2 (model training)
